In [6]:
import torch
import numpy as np
import torch.nn.functional as F
from ai_engine import load_model_pipeline
from text_preprocessor import TextPreprocessor
from embed_utils import embed_text, embed_combined
from graph_utils import build_star_graph

# ───────────────────────────────────────────────
# 0. โหลด Pipeline
# ───────────────────────────────────────────────
pipeline = load_model_pipeline()
tokenizer  = pipeline["tokenizer"]
bert_model = pipeline["bert_model"]
model      = pipeline["model"]
nbrs       = pipeline["nbrs"]
x_database = pipeline["x_database"]
id2label   = pipeline["id2label"]
device     = pipeline["device"]
k          = pipeline["k_neighbors"]

raw_text = "รัฐบาลแจกเงิน 1 ล้านบาทให้ประชาชนทุกคนฟรีทันที"

# ───────────────────────────────────────────────
# 1. Preprocessing
# ───────────────────────────────────────────────
cleaned, ok, msg = TextPreprocessor.preprocess(raw_text)
print(f"[PREPROCESS] Valid: {ok}")
print(f"[PREPROCESS] Cleaned: {cleaned}")
print(f"[PREPROCESS] Length: {len(cleaned)} chars, {len(cleaned.split())} words")

# ───────────────────────────────────────────────
# 2. Tokenization
# ───────────────────────────────────────────────
inputs = tokenizer(cleaned, return_tensors="pt", truncation=True, max_length=256, padding=True).to(device)
print(f"\n[TOKENS] Input IDs: {inputs['input_ids'][0][:20].tolist()}...")
print(f"[TOKENS] Token Count (real): {inputs['attention_mask'].sum().item()}")
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0][:20].tolist())
print(f"[TOKENS] Decoded: {tokens}")

# ───────────────────────────────────────────────
# 3. BERT Embedding
# ───────────────────────────────────────────────
with torch.inference_mode():
    outputs = bert_model(**inputs)
lhs = outputs.last_hidden_state
print(f"\n[BERT] Last Hidden State shape: {lhs.shape}")  # (1, seq_len, 768)
print(f"[BERT] CLS token vector (first 5 dims): {lhs[0][0][:5].tolist()}")

# Mean Pool + L2 Normalize
emb = embed_text(cleaned, tokenizer, bert_model, device)
print(f"\n[EMBEDDING] Shape: {emb.shape}")  # (768,)
print(f"[EMBEDDING] L2 Norm: {np.linalg.norm(emb):.6f}")  # ≈ 1.000
print(f"[EMBEDDING] First 5 dims: {emb[:5].tolist()}")

# ───────────────────────────────────────────────
# 4. kNN Search
# ───────────────────────────────────────────────
dists, idxs = nbrs.kneighbors(emb.reshape(1, -1), n_neighbors=k)
dists, idxs = dists[0], idxs[0]
print(f"\n[kNN] Top-{k} neighbor indices: {idxs.tolist()}")
print(f"[kNN] Cosine distances: {dists.round(4).tolist()}")
print(f"[kNN] Cosine similarities: {(1-dists).round(4).tolist()}")

# ───────────────────────────────────────────────
# 5. Star Graph
# ───────────────────────────────────────────────
neighbor_embs = x_database[idxs]
graph = build_star_graph(emb, neighbor_embs, dists, device)
print(f"\n[GRAPH] Nodes: {graph.x.shape}")        # (11, 768)
print(f"[GRAPH] Edge index shape: {graph.edge_index.shape}")  # (2, 21)
print(f"[GRAPH] Edge weights: {graph.edge_attr.cpu().numpy().round(4).tolist()}")
print(f"[GRAPH] Self-loop weight: {graph.edge_attr[-1].item():.2f}")  # = 1.0

# ───────────────────────────────────────────────
# 6. GCN Inference
# ───────────────────────────────────────────────
with torch.no_grad():
    logits_all = model(graph)               # (11, 2)
    query_logits = logits_all[0]            # (2,) — เฉพาะ node 0
    probs = F.softmax(query_logits, dim=0)  # (2,)

print(f"\n[GCN] All node logits shape: {logits_all.shape}")  # (11, 2)
print(f"[GCN] Query logits (raw): {query_logits.cpu().numpy().round(4).tolist()}")
print(f"[GCN] Probabilities: Fake={probs[0].item():.4f}, Real={probs[1].item():.4f}")

pred_idx   = int(torch.argmax(probs).item())
confidence = float(probs[pred_idx]) * 100
label      = id2label[pred_idx]
result     = "Real" if "จริง" in label else "Fake"

print(f"\n[RESULT] ============================================")
print(f"[RESULT] Prediction:  {result}")
print(f"[RESULT] Confidence:  {confidence:.2f}%")
print(f"[RESULT] Thai Label:  {label}")
print(f"[RESULT] ============================================")

2026-03-23 21:05:51.774 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3583.84it/s]
CamembertModel LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Users\tt_pe\Documents\GitHub\Project_Thaifakenews\ai_engine.py:96: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbit

[PREPROCESS] Valid: True
[PREPROCESS] Cleaned: รัฐบาลแจกเงิน 1 ล้านบาทให้ประชาชนทุกคนฟรีทันที
[PREPROCESS] Length: 46 chars, 3 words

[TOKENS] Input IDs: [5, 9849, 17541, 10, 59, 10, 548, 2031, 343, 1069, 1013, 6]...
[TOKENS] Token Count (real): 12
[TOKENS] Decoded: ['<s>', '▁รัฐบาล', 'แจกเงิน', '▁', '1', '▁', 'ล้านบาท', 'ให้ประชาชน', 'ทุกคน', 'ฟรี', 'ทันที', '</s>']

[BERT] Last Hidden State shape: torch.Size([1, 12, 768])
[BERT] CLS token vector (first 5 dims): [-0.6692408919334412, 1.052799940109253, 0.6257883906364441, 1.3392874002456665, 1.445851445198059]

[EMBEDDING] Shape: (768,)
[EMBEDDING] L2 Norm: 1.000000
[EMBEDDING] First 5 dims: [-0.007766182534396648, 0.03528526797890663, 0.006829524878412485, 0.04987074062228203, 0.04358796030282974]

[kNN] Top-10 neighbor indices: [4922, 6598, 1392, 679, 4305, 2414, 3349, 8782, 6219, 986]
[kNN] Cosine distances: [0.29589998722076416, 0.3012000024318695, 0.3077000081539154, 0.31290000677108765, 0.3188000023365021, 0.33219999074935913, 0